#学習用イベント分類モデル（DASEventNet）トレーニング
このノートブックでは、Yu et al.（2024）に基づき、DAS波形データにおける地震イベントとノイズの分類モデル「DASEventNet」のトレーニングを実施する。

処理内容：


*   ResNet-50 をベースとしたノイズ・イベント2クラス分類モデルを構築（2秒のDASデータを入力）
*   Silixa社が公開した1,309件のイベントをもとに、学習・検証・テストに時系列順でイベントを分割（それぞれ 75% / 15% / 10%）
*   イベントとノイズのデータの割合が1:1となるようにノイズデータを追加しデータセットを構築
*   BCEWithLogitsLoss により学習され、検証損失の最小化に基づいて Early Stopping を導入
*   Colab のリソース制限に対応するため、以下を導入：

  ・使用データは単精度(float16)に変換

  ・混合精度学習（AMP） によるメモリ効率化

  ・memmap形式 による大規模データの逐次読み込み

In [1]:
!pip install -U gdown
!pip install -q torch torchvision torchaudio tensorboard
from pathlib import Path
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import GradScaler, autocast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.1 MB/s eta 0:00:00


#イベント・ノイズデータの読み込み

In [2]:
#googleドライブからデータをダウンロード
import gdown
output = "event.npy"
file_id = '14yi7YV7R0Dz3xkhZh6AYvyQitiSLCiTv'
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)
event_data = np.load(output, mmap_mode='r')
print(event_data.shape)
print(event_data.dtype)

output = "noise.npy"
file_id = '1r7Wre9vUBXDcCQlMEBaEGpsTy6ertAAp'
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)
noise_data = np.load(output, mmap_mode='r')
print(noise_data.shape)


Downloading...
From (original): https://drive.google.com/uc?id=14yi7YV7R0Dz3xkhZh6AYvyQitiSLCiTv
From (redirected): https://drive.google.com/uc?id=14yi7YV7R0Dz3xkhZh6AYvyQitiSLCiTv&confirm=t&uuid=1f008a72-f24a-4ca5-81a2-0809b02c2be4
To: /content/event.npy
100%|██████████| 5.35G/5.35G [01:18<00:00, 67.9MB/s]


(1309, 1021, 2000)
float16


Downloading...
From (original): https://drive.google.com/uc?id=1r7Wre9vUBXDcCQlMEBaEGpsTy6ertAAp
From (redirected): https://drive.google.com/uc?id=1r7Wre9vUBXDcCQlMEBaEGpsTy6ertAAp&confirm=t&uuid=cc2f51f2-0e17-4b79-a8d6-122a3b1da9ed
To: /content/noise.npy
100%|██████████| 5.35G/5.35G [01:17<00:00, 68.9MB/s]

(1309, 1021, 2000)


イベントは発生時刻の早い順に、以下の割合で3つのデータセットに分割：
- トレーニングデータ：75%（986件）
- 検証データ：15%（193件）
- テストデータ：10%（130件）
さらに、それぞれのデータセットには、対応する件数のノイズデータも加え、イベントとノイズの比率が1:1になるように構成。

In [3]:
# ダウンロードデータは時系列順にデータが並んでいる
# 新しいイベントが前に来るように、イベントの順番を逆にする
event_data = event_data[::-1]
noise_data = noise_data[::-1]

n=0
test_range = np.arange(0, 130)  # 10%
valid_range = np.arange(130, 130 + 193)  # 15%
train_range = np.arange(130 + 193, len(event_data))  # 75%


答えのデータはイベントが１でノイズが０の2値ラベル

In [8]:
batch_size = 4

class WaveformDataset(Dataset):
    def __init__(self,event_data, noise_data, idx_range):
        self.event = event_data
        self.noise = noise_data
        self.idx_range = idx_range
    def __len__(self):
        return (len(self.idx_range))*2

    def __getitem__(self, idx):
        if idx < len(self.idx_range): # choose event
          x = self.event[self.idx_range[idx]].astype(np.float32)  # (H, W)
          label=0
        else:
          j = idx - len(self.idx_range)
          x = self.noise[self.idx_range[j]].astype(np.float32)
          label=1
        # 標準化（行方向 = trace ごと）
        mean = np.mean(x, axis=1, keepdims=True)
        std  = np.std(x, axis=1, keepdims=True) + 1e-10  # avoid div‑by‑zero
        x = (x - mean) / std
        x = torch.from_numpy(x).unsqueeze(0)  # (1, H, W)
        y = torch.tensor(label, dtype=torch.float32)
        return x, y

train_dataset = WaveformDataset(event_data, noise_data, train_range)
valid_dataset = WaveformDataset(event_data, noise_data, valid_range)
test_dataset = WaveformDataset(event_data, noise_data, test_range)

train_loader = DataLoader(
	train_dataset, batch_size=batch_size, shuffle=True, num_workers=4
)
valid_loader = DataLoader(
	valid_dataset, batch_size=batch_size, shuffle=False, num_workers=4
)
test_loader = DataLoader(
	test_dataset, batch_size=batch_size, shuffle=False, num_workers=4
)



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


## 学習モデルの構築
 ResNet-50を使用

In [9]:
def build_model():
    model = resnet50(weights=None)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Sequential(nn.Flatten(), nn.Linear(2048, 1))
    return model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model().to(device)

total_params = sum(p.numel() for p in model.parameters())
# 学習可能なパラメータ数（requires_grad=True のみ）
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'総パラメータ数: {total_params:,}')
print(f'学習可能パラメータ数: {trainable_params:,}')

総パラメータ数: 23,503,809
学習可能パラメータ数: 23,503,809


## 学習の実施

In [10]:
def lr_schedule(epoch: int, base_lr: float = 1e-4):
	if epoch > 20:
		return base_lr * 0.1
	if epoch > 10:
		return base_lr * 0.5
	return base_lr

def train(
	model: nn.Module,
	loaders: tuple[DataLoader, DataLoader],
	device: torch.device,
	save_dir: Path,
	epochs: int = 5,
	patience: int = 5,
):
	train_loader, val_loader = loaders
	criterion = nn.BCEWithLogitsLoss()
	optimizer = optim.Adam(model.parameters(), lr=lr_schedule(0))
	scaler = GradScaler()
	writer = SummaryWriter(log_dir=str(save_dir / 'tb'))

	best_val_loss = float('inf')
	epochs_no_improve = 0

	for epoch in range(epochs):
		# LR scheduling
		lr = lr_schedule(epoch)
		for pg in optimizer.param_groups:
			pg['lr'] = lr
		writer.add_scalar('LR', lr, epoch)

		# ---------- Training ----------
		model.train()
		train_loss, train_correct = 0.0, 0
		for x, y in train_loader:
			x, y = x.to(device), y.to(device).unsqueeze(1)
			optimizer.zero_grad()
			with autocast():
				logits = model(x)
				loss = criterion(logits, y)
			scaler.scale(loss).backward()
			scaler.step(optimizer)
			scaler.update()

			train_loss += loss.item() * x.size(0)
			preds = torch.sigmoid(logits) >= 0.5
			train_correct += (preds == y.bool()).sum().item()

		train_loss /= len(train_loader.dataset)
		train_acc = train_correct / len(train_loader.dataset)

		# ---------- Validation ----------
		model.eval()
		val_loss, val_correct = 0.0, 0
		with torch.no_grad():
			for x, y in val_loader:
				x, y = x.to(device), y.to(device).unsqueeze(1)
				logits = model(x)
				loss = criterion(logits, y)
				val_loss += loss.item() * x.size(0)
				preds = torch.sigmoid(logits) >= 0.5
				val_correct += (preds == y.bool()).sum().item()

		val_loss /= len(val_loader.dataset)
		val_acc = val_correct / len(val_loader.dataset)

		# ---------- Logging ----------
		writer.add_scalars('Loss', {'Train': train_loss, 'Val': val_loss}, epoch)
		writer.add_scalars('Accuracy', {'Train': train_acc, 'Val': val_acc}, epoch)

		print(
			f'Epoch {epoch + 1:03d}/{epochs} – loss: {train_loss:.4f} – val_loss: {val_loss:.4f} – acc: {train_acc:.4f} – val_acc: {val_acc:.4f}'
		)

		# Early stopping & checkpoint
		if val_loss < best_val_loss:
			best_val_loss = val_loss
			epochs_no_improve = 0
			torch.save(model.state_dict(), save_dir / 'best_model.pth')
		else:
			epochs_no_improve += 1
			if epochs_no_improve >= patience:
				print('Early stopping triggered.')
				break

	writer.close()

save_dir=Path('output')
train(model, (train_loader, valid_loader), device, save_dir)



/tmp/ipython-input-10-427615339.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-10-427615339.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001/100 – loss: 0.1860 – val_loss: 0.0585 – acc: 0.9336 – val_acc: 0.9974
Epoch 002/100 – loss: 0.0758 – val_loss: 0.3157 – acc: 0.9741 – val_acc: 0.8316
Epoch 003/100 – loss: 0.0454 – val_loss: 0.1371 – acc: 0.9878 – val_acc: 0.9482
Epoch 004/100 – loss: 0.0414 – val_loss: 0.1507 – acc: 0.9883 – val_acc: 0.9663
Epoch 005/100 – loss: 0.0244 – val_loss: 0.0268 – acc: 0.9914 – val_acc: 0.9948


KeyboardInterrupt: 

##テストデータの評価

In [11]:
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device):
	criterion = nn.BCEWithLogitsLoss()
	model.eval()
	loss, correct = 0.0, 0
	with torch.no_grad():
		for x, y in loader:
			x, y = x.to(device), y.to(device).unsqueeze(1)
			logits = model(x)
			loss += criterion(logits, y).item() * x.size(0)
			preds = torch.sigmoid(logits) >= 0.5
			correct += (preds == y.bool()).sum().item()
	loss /= len(loader.dataset)
	acc = correct / len(loader.dataset)
	return loss, acc


# Load best model for evaluation
model.load_state_dict(torch.load( save_dir / 'best_model.pth', map_location=device))

for split_name, loader in [
	('train', train_loader),
	('test', test_loader),
	('valid', valid_loader),
]:
	loss, acc = evaluate(model, loader, device)
	print(f'{split_name.capitalize()} – Loss: {loss:.4f} | Acc: {acc:.4f}')


Train – Loss: 0.0228 | Acc: 0.9975
Test – Loss: 0.0245 | Acc: 0.9962
Valid – Loss: 0.0268 | Acc: 0.9948


注記：Colab用に学習パラメータを調整したので、実際の再現コードの結果と完全には一致しない